In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Patients Notebook").getOrCreate()

In [0]:
# Read the Vitals CSV file into a Spark DataFrame
bronze_path = r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/bronze/dbo.patients.csv'
patients_bronze_df = spark.read.csv(bronze_path, header = True, inferSchema = True)

# Vital raw rows and columns counts
print(f"Patients file has {patients_bronze_df.count()} rows and {len(patients_bronze_df.columns)} columns")
patients_bronze_df.printSchema()

display(patients_bronze_df)

In [0]:
# -------------------------------------------
# 1. Deduplication based on "patient_id" column (Primary Key)
# -------------------------------------------

patients_deduplicate_df = patients_bronze_df.dropDuplicates(["patient_id"])
print(f"After deduplication: {patients_deduplicate_df.count()} rows")
print(f"Duplicate records count is {patients_bronze_df.count() - patients_deduplicate_df.count()}")
display(patients_deduplicate_df)

In [0]:
# -----------------------------
# 2. Date Standardization
# -----------------------------


# A. Parse ingestion_timestamp to a proper timestamp

from pyspark.sql.functions import coalesce, col, expr, when

timestamp_df = patients_deduplicate_df.withColumn(
    "ingestion_ts_parsed",
    coalesce(
        expr("try_to_timestamp(ingestion_timestamp, 'dd-MM-yyyy HH:mm')"),
        expr("try_to_timestamp(ingestion_timestamp, 'M/dd/yy H:mm')"),
        expr("try_to_timestamp(ingestion_timestamp, 'MM/dd/yyyy HH:mm')"),
        expr("try_to_timestamp(ingestion_timestamp, 'yyyy-MM-dd HH:mm:ss')"),
    )
)
#display(timestamp_df)

# B. Date of Birth — Multi-format Parse 
patients_dob_parsed_df = timestamp_df.withColumn(
    "dob_parsed",
    coalesce(
        expr("try_to_date(date_of_birth, 'dd-MM-yyyy')"),
        expr("try_to_date(date_of_birth, 'yyyy-MM-dd')"),
        expr("try_to_date(date_of_birth, 'MM-dd-yyyy')"),
        expr("try_to_date(date_of_birth, 'dd/MM/yyyy')"),
        expr("try_to_date(date_of_birth, 'MM/dd/yyyy')"),
        expr("try_to_date(date_of_birth, 'MM-dd-yy')")

    )
)
            # DQ flag: 1 if date_of_birth provided but could not be parsed
patients_dob_parsed_df1 = patients_dob_parsed_df.withColumn(
    "dq_dob",
    when(col("date_of_birth").isNotNull() & col("dob_parsed").isNull(), 1).otherwise(0)
)
#display(patients_dob_parsed_df1)

#--------------------------------------------------------------------------------------------------------

# C. Registration Date Standardisation
patients_registration_date_df = patients_dob_parsed_df1.withColumn(
 "registration_date_std",
    coalesce(
        expr("try_to_date(registration_date, 'dd-MM-yyyy')"),
        expr("try_to_date(registration_date, 'yyyy-MM-dd')"),
        expr("try_to_date(registration_date, 'MM-dd-yy')"),
        expr("try_to_date(registration_date, 'MM/dd/yyyy')"),
        expr("try_to_date(registration_date, 'MM/dd/yy')"),
    )
)
#display(patients_registration_date_df)

In [0]:
# -----------------------------
# 3. Casing Standardization
# -----------------------------

from pyspark.sql.functions import initcap, trim, col, when, lit, concat_ws, upper
from pyspark.sql.types import StringType

# A. Name Cleaning & Full Name 
patients_fullname_df = ( patients_registration_date_df
    .withColumn("first_name_clean", initcap(trim(col("first_name")))) # first_name_clean: trim +title case
    .withColumn("last_name_clean",  initcap(trim(col("last_name"))))  # last_name_clean: trim + title case
    .withColumn(
        "full_name",
        when( 
             col("first_name_clean").isNull() & col("last_name_clean").isNull(),
            lit(None).cast(StringType())
        ).otherwise(
            concat_ws(" ", col("first_name_clean"), col("last_name_clean"))
        )
    )
)
#display(patients_fullname_df)

#---------------------------------------------------------------------------------------------------------------

# B. City & State — Casing 

patients_city_state_df = ( patients_fullname_df
    .withColumn("city_clean",  initcap(trim(col("city"))))
    .withColumn("state_clean", initcap(trim(col("state"))))
)
#display(patients_city_state_df)

#---------------------------------------------------------------------------------------------------------------

# C. Source System — UPPER + NULL handling ────────────────────────────────

patients_source_system_df= patients_city_state_df.withColumn(
    "source_system_upper",
    when(
        col("source_system").isNull() | (trim(col("source_system")) == ""),
        lit("UNKNOWN")
    ).otherwise(upper(trim(col("source_system"))))
)
display(patients_source_system_df)

In [0]:
# -----------------------------
# 4. lookup/Standardization
# -----------------------------

# A. Gender Standardisation (case-insensitive) 
patients_genderstd_df = patients_source_system_df.withColumn(
    "gender_std",
    when(upper(trim(col("gender"))).isin("M", "MALE"),"Male")
    .when(upper(trim(col("gender"))).isin("F", "FEMALE"),"Female")
    .when(upper(trim(col("gender"))).isin("OTHER"),"Other")
    .when(col("gender").isNull(), "Unknown")
    .otherwise( "Unknown")
)
#display(patients_genderstd_df)

#-----------------------------------------------------------------------------------------------------

# B. Blood Group Standardisation

blood_col = upper(trim(col("blood_group")))          # Clean the column once
null_values = ["N/A", "NA", "NONE", "NULL", ""]      # List of invalid/null values

blood_group_map = {                                  # All valid mappings in one simple dictionary
    "A POSITIVE":  "A+",                 
    "A NEGATIVE":  "A-",
    "B POSITIVE":  "B+",
    "B NEGATIVE":  "B-",
    "AB POSITIVE": "AB+",
    "AB NEGATIVE": "AB-",
    "O POSITIVE":  "O+",
    "O NEGATIVE":  "O-",
    "A+": "A+", "A-": "A-",
    "B+": "B+", "B-": "B-",
    "AB+": "AB+", "AB-": "AB-",
    "O+": "O+", "O-": "O-",
}

blood_expr = when(blood_col.isin(null_values), lit(None).cast(StringType())) # If value is null/invalid → return None

for raw, std in blood_group_map.items():                                     # Map each known value to its standard form
    blood_expr = blood_expr.when(blood_col == raw, lit(std))

blood_expr = blood_expr.otherwise(lit(None).cast(StringType()))              # Anything else not in the map → return None   

patients_blood_group_std_df = patients_genderstd_df.withColumn("blood_group_std", blood_expr)  # Add the new standardised column

#display(patients_blood_group_std_df)

#------------------------------------------------------------------------------------------------------

# C. Marital Status Standardisation (case-insensitive) 

patients_marrige_std_df = patients_blood_group_std_df.withColumn(
    "marital_status_std",
    when(upper(trim(col("marital_status"))) == "MARRIED","Married")
    .when(upper(trim(col("marital_status"))).isin("SINGLE","UNMARRIED"),"Single")
    .when(upper(trim(col("marital_status"))) == "DIVORCED","Divorced")
    .when(upper(trim(col("marital_status"))) == "WIDOWED","Widowed")
    .when(col("marital_status").isNull(),"Unknown")
    .otherwise("Unknown")
)
display(patients_marrige_std_df)



In [0]:
# -----------------------------
# 5. Validation
# -----------------------------

from pyspark.sql.functions import length
from pyspark.sql.types import IntegerType

# A. Email Validation
patients_email_df = patients_marrige_std_df.withColumn(
    "email_valid",
    when(
        col("email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"),col("email")
    ).otherwise(lit(None).cast(StringType()))
)
#display(patients_email_df)

#--------------------------------------------------------------------------------------------------------

# B. Pincode Validation (6-digit Indian PIN) 

patients_pincode_df = patients_email_df.withColumn(
    "pincode_validated",
    when(
        length(col("pincode").cast(StringType())) == 6,
        col("pincode").cast(IntegerType())
    ).otherwise(lit(None).cast(IntegerType()))
)
#display(patients_pincode_df)

#----------------------------------------------------------------

In [0]:
# -----------------------------------------------------------
# 6. Phone number Normalisation and Boolean Normalisation
# -----------------------------------------------------------

from pyspark.sql.functions import regexp_extract, regexp_replace, col, when, lit
from pyspark.sql.types import StringType

# A. Phone Normalisation (10-digit Indian mobile) 
patients_phone_df = ( patients_pincode_df      # Strip everything non-digit, then extract exactly 10 consecutive digits
    .withColumn("_phone_digits", regexp_extract(
        regexp_replace(col("phone_primary"), r"[^\d]", ""),r"(\d{10})",1
    )
 )
    .withColumn(
        "phone_std",
        when(col("_phone_digits") == "", lit(None).cast(StringType()))
        .otherwise(col("_phone_digits"))
    )
    .drop("_phone_digits")
)

#display(patients_phone_df)

# B. is_active Boolean Normalisation 
patients_booleanflag_df = patients_phone_df.withColumn(
    "is_active_flag",
    when(
        upper(trim(col("is_active").cast(StringType()))).isin("Y", "YES", "1", "TRUE"), lit(1))
    .when(
        upper(trim(col("is_active").cast(StringType()))).isin("N", "NO", "0", "FALSE"), lit(0))
    .otherwise(lit(None).cast(IntegerType())).cast(IntegerType())
)
display(patients_booleanflag_df)


In [0]:
# -----------------------------
# 7. Derived Columns
# -----------------------------

from pyspark.sql.functions import floor, datediff, current_date, col, when, lit, to_date
from pyspark.sql.types import IntegerType

# A. Age in Years 
patients_age_years_df = patients_booleanflag_df.withColumn(
    "age_years",
    when(
        col("dob_parsed").isNotNull(),
        floor(datediff(current_date(), col("dob_parsed")) / 365.25).cast(IntegerType())
    ).otherwise(lit(None).cast(IntegerType()))
)
#display(patients_age_years_df)

#--------------------------------------------------------------------------------------------------

# B. Age Group 
patients_age_group_df = patients_age_years_df.withColumn(
    "age_group",
    when(col("age_years").isNull(),"Unknown")
    .when(col("age_years") < 18,"Pediatric")
    .when(col("age_years") < 60,"Adult")
    .otherwise("Senior")
)
#display(patients_age_group_df)

#------------------------------------------------------------------------------------------------

# C. Ingestion Date (partition key)
patients_ingestion_date_df = patients_age_group_df.withColumn(
    "ingestion_date",
    to_date(col("ingestion_ts_parsed"))
)
display(patients_ingestion_date_df)



In [0]:
# -----------------------------
# 8. Data Quality Checks Score
# -----------------------------

# Each check returns 1 (clean) or 0 (issue)

from pyspark.sql.functions import col, when, lit, round 

patient_id_check   = when(col("patient_id").isNotNull(),           1).otherwise(0)
full_name_check    = when(col("full_name").isNotNull(),             1).otherwise(0)
dob_check          = when(col("dob_parsed").isNotNull(),            1).otherwise(0)
age_check          = when(col("age_years").isNotNull(),             1).otherwise(0)
gender_check       = when(col("gender_std") != "Unknown",           1).otherwise(0)
blood_group_check  = when(col("blood_group_std").isNotNull(),       1).otherwise(0)
phone_check        = when(col("phone_std").isNotNull(),             1).otherwise(0)
email_check        = when(col("email_valid").isNotNull(),           1).otherwise(0)
pincode_check      = when(col("pincode_validated").isNotNull(),     1).otherwise(0)
reg_date_check     = when(col("registration_date_std").isNotNull(), 1).otherwise(0)


all_checks = [
    patient_id_check,
    full_name_check,
    dob_check,
    age_check,
    gender_check,
    blood_group_check,
    phone_check,
    email_check,
    pincode_check,
    reg_date_check,
]

Total_checks = len(all_checks)  
total_passed = sum(all_checks)   
dq_score_expr = round((total_passed / lit(Total_checks)) * 100, 2)

patients_dq_score_df = patients_ingestion_date_df.withColumn("dq_score", dq_score_expr)
display(patients_dq_score_df)


patients_silver_df = patients_dq_score_df

In [0]:
# ----------------------------------------
# 9. Write to Silver layer in delta format
# ----------------------------------------


TABLE_NAME = "patients"  

silver_table_path = f"abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/silver/{TABLE_NAME}"

# ── OVERWRITE (Full Load) ────────────────────────────────────
patients_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_table_path)

print(f"✅ Data written to Silver layer: {silver_table_path}")
